# Project 2

### Group J

In [ ]:
# Imports
from datetime import datetime, timezone
import os
import json

spark_ok = True
try:
    import pyspark.sql.functions as F
    from pyspark.sql import SparkSession
    from pyspark.context import SparkContext
except Exception as e:
    spark_ok = False
    print("PySpark not available:", e)

In [2]:
# Init Spark

sc = SparkContext('local', 'project_1')

spark = (
    SparkSession.builder
    .appName('project_1')
    .getOrCreate()
)

print(spark.version)

4.1.0


In [6]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Packages are loaded via PYSPARK_SUBMIT_ARGS set in compose.yml.
# pyspark-notebook:2025-12-31 ships Spark 4.1.0 — print spark.version to confirm.

spark = (
    SparkSession.builder
    .appName("project2")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")

    # ── Iceberg ──────────────────────────────────────────────────────────────
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # Catalog named 'lakehouse' — use it as: lakehouse.<database>.<table>
    .config("spark.sql.catalog.lakehouse",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type",      "rest")
    .config("spark.sql.catalog.lakehouse.uri",       "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    # S3FileIO writes data files directly to MinIO
    .config("spark.sql.catalog.lakehouse.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint",          "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id",     os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")

    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}   catalog: lakehouse")

# ── Create your database once ──────────────────────────────────────────────
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.taxi")

Spark 4.1.0   catalog: lakehouse


DataFrame[]

In [7]:
BOOTSTRAP = "kafka:9092"
TOPIC     = "taxi-trips"

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

### In-Class Project Task

#### Create the Bronze Table

In [20]:
spark.sql("SHOW CATALOGS").show()

+-------------+
|      catalog|
+-------------+
|    lakehouse|
|spark_catalog|
+-------------+



In [21]:
# Create Bronze Table
spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.taxi.bronze (
    json STRING
) USING iceberg
""")

DataFrame[]

#### Simple streaming query (Kafka to Bronze)

In [45]:
# Streaming query
df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "taxi-trips")
    .option("startingOffsets", "earliest")
    .option("kafka.group.id", "spark-group")
    .load()
)

df = df.selectExpr("CAST(value AS STRING) as json")

query = (
    df.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/checkpoints/bronze")
    .toTable("lakehouse.taxi.bronze")
)

In [43]:
# Stop the query after a bit
query.stop()

#### Query the bronze table

In [39]:
spark.sql("SELECT count(*) FROM lakehouse.taxi.bronze").show()

+--------+
|count(1)|
+--------+
|     549|
+--------+



In [40]:
spark.sql("SELECT * FROM lakehouse.taxi.bronze LIMIT 3").show(truncate=False)

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|json                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                |
+-------------------------

##### We tried to find the group, but not even the 'enable.auto.commit = true' option didn't help us find it. 